# Minimal ReAct Agent with LangChain `create_agent` + Groq

In [ ]:
# %pip install -q langchain langchain-groq langchain-core langgraph

## 1.Access Model API key

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
GROQ_API_KEY  = os.getenv("GROQ_API_KEY")

#print(GROQ_API_KEY)

## 2.Create react agent with llm, systemprompt, tools 

In [2]:
from langchain.agents import create_agent         
from langchain_groq import ChatGroq
from langchain_core.tools import tool

GROQ_MODEL = "qwen/qwen3-32b"

# --- Tools ---
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    if location.lower() in ["sf", "san francisco"]:
        return "It's 60 degrees and foggy."
    return "It's 90 degrees and sunny."

@tool
def get_coolest_cities() -> str:
    """Get a list of the coolest cities."""
    return "nyc, sf"

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

tools = [get_weather, get_coolest_cities, multiply]

# --- Model ---
llm = ChatGroq(model=GROQ_MODEL)

# --- System Prompt ---
systemPrompt ="You are a helpful assistant. Use tools when needed to answer the user's question."

# --- Agent ---
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt = systemPrompt,
)

## 3.Run Agent (as a loop)

In [3]:
# --- Run ---
user_question = "Will it rain tomorrow morning in Berlin?"

response = agent.invoke({
    "messages": [("user", user_question)]
})

# Final answer
print(response["messages"][-1].content)

The weather in Berlin tomorrow morning will be sunny with a temperature of 90°F (32°C). There is no indication of rain.


In [4]:
# Optional: inspect full message trace (tool calls + results)
for msg in response["messages"]:
    print(f"[{msg.__class__.__name__}] {msg.content}")

[HumanMessage] Will it rain tomorrow morning in Berlin?
[AIMessage] 
[ToolMessage] It's 90 degrees and sunny.
[AIMessage] The weather in Berlin tomorrow morning will be sunny with a temperature of 90°F (32°C). There is no indication of rain.


In [5]:
from langchain.agents.middleware import AgentMiddleware


class TrajectoryLoggerMiddleware(AgentMiddleware):

    def before_model(self, state, runtime):
        print("\n================ MODEL =================")
        print("Messages sent to model:")
        print(state["messages"][-1])

    def after_model(self, state, runtime):
        print("\n================ MODEL RESPONSE =================")
        print(state["messages"][-1])

    def before_tool(self, tool_call, runtime):
        print("\n================ TOOL CALL =================")
        print(f"Tool: {tool_call['name']}")
        print(f"Args: {tool_call['args']}")

    def after_tool(self, tool_call, result, runtime):
        print("\n================ TOOL RESULT =================")
        print(result)


agent_debug = create_agent(
    model=llm,
    tools=tools,
    system_prompt=systemPrompt,
    middleware=[
        TrajectoryLoggerMiddleware()
    ]
)

In [6]:
result = agent_debug.invoke(
    {
        "messages": [
            ("user", "What is 25 * 4?")
        ]
    }
)


================ MODEL =================
Messages sent to model:
content='What is 25 * 4?' additional_kwargs={} response_metadata={} id='510cecac-6302-459e-ad1b-102910075035'

================ MODEL RESPONSE =================
content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking "What is 25 * 4?" So I need to calculate that. Let me check the tools available. There\'s a multiply function called \'multiply\' that takes two integers, a and b. The parameters are required, so I need to provide both. The user\'s question is straightforward: 25 multiplied by 4. So the arguments should be a=25 and b=4. I should call the multiply function with these values. I don\'t need any other tools here since it\'s a simple multiplication. Let me make sure the parameters are correct. Yes, both are integers. Alright, time to format the tool call.\n', 'tool_calls': [{'id': 've2fmjqk0', 'function': {'arguments': '{"a":25,"b":4}', 'name': 'multiply'}, 'type': 'function'}]} response_meta

In [ ]:
# !pip install agentevals

In [7]:
def print_trajectory(messages):
    print("\n===== TRAJECTORY =====")

    for i, msg in enumerate(messages):
        print(f"\n[{i}] {msg.__class__.__name__}")
        print(msg.content)

        if hasattr(msg, "tool_calls"):
            print("tool_calls =", msg.tool_calls)

## 2.1 Trajectory Match Evaluator

In [8]:
from agentevals.trajectory.match import (
    create_trajectory_match_evaluator
)

trajectory_evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="strict"
)

outputs = agent.invoke(
    {
        "messages": [
            ("user", "What is 3 * 5?")
        ]
    }
)

print_trajectory(outputs["messages"])

reference_outputs = {
    "messages": outputs["messages"]
}


===== TRAJECTORY =====

[0] HumanMessage
What is 3 * 5?

[1] AIMessage

tool_calls = [{'name': 'multiply', 'args': {'a': 3, 'b': 5}, 'id': 'k8vd659pm', 'type': 'tool_call'}]

[2] ToolMessage
15

[3] AIMessage
The result of multiplying 3 by 5 is **15**.
tool_calls = []


In [9]:
score = trajectory_evaluator(
    outputs=outputs["messages"],
    reference_outputs=reference_outputs["messages"]
)

print(score)

{'key': 'trajectory_strict_match', 'score': True, 'comment': None, 'metadata': None}


## 2.2 LLM-as-Judge

In [14]:
from agentevals.trajectory.llm import (
    create_trajectory_llm_as_judge,
    TRAJECTORY_ACCURACY_PROMPT
)

trajectory_judge = create_trajectory_llm_as_judge(
    model="groq:qwen/qwen3-32b"  # <-- Use the "groq:" prefix string here
)

In [15]:
outputs = agent.invoke(
    {
        "messages": [
            ("user", "What is the weather in SF?")
        ]
    }
)

print_trajectory(outputs["messages"])

evaluation = trajectory_judge(
    outputs=outputs["messages"]
)

print("\n===== JUDGE RESULT =====")
print(evaluation)


===== TRAJECTORY =====

[0] HumanMessage
What is the weather in SF?

[1] AIMessage

tool_calls = [{'name': 'get_weather', 'args': {'location': 'San Francisco'}, 'id': 'nsbd63gkh', 'type': 'tool_call'}]

[2] ToolMessage
It's 60 degrees and foggy.

[3] AIMessage
The weather in San Francisco is currently 60°F and foggy. 🌫️ It's a good day to carry an umbrella just in case!
tool_calls = []

===== JUDGE RESULT =====
{'key': 'trajectory_accuracy', 'score': True, 'comment': "The trajectory logically follows the user's query by calling the 'get_weather' function with the correct location. The response accurately reflects the tool's output and adds a relevant, context-appropriate suggestion about carrying an umbrella. The steps show clear progression and efficiency. Thus, the score should be: true.", 'metadata': None}


## 3. Middleware Showcase

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.agents import create_agent         

# 1. Define the policy
hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "multiply": True,
        "get_weather": True,
    }
)

# 2. Compile the correct agent with the checkpointer attached
agent_hitl = create_agent(
    model=llm,
    tools=tools,
    checkpointer=InMemorySaver(), 
    middleware=[hitl]
)

# 3. Setup configurations
config = {"configurable": {"thread_id": "demo-hitl-session"}}
initial_input = {"messages": [("user", "What is the current weather in San Francisco right now?")]}

print("🚀 Starting Agent Execution...")
# First invocation: runs up to the get_weather tool and halts
response = agent_hitl.invoke(initial_input, config=config)

# --- 4. Read the True Graph State to check for an active Interruption ---
current_graph_state = agent_hitl.get_state(config)

if current_graph_state.next:  # If there are next steps but execution paused, we are interrupted!
    print("\n🛑 [HITL INTERRUPT TRIGGERED] The agent is paused awaiting human review!")
    
    # Extract the details of the pending tool call intercepted by the middleware
    # LangGraph structures active interrupts under the tasks metadata or state.tasks
    active_tasks = current_graph_state.tasks
    if active_tasks and active_tasks[0].interrupts:
        interrupt_info = active_tasks[0].interrupts[0]
        
        # Access the proposed tool and its args from the HITL middleware request
        print(f"👉 Requested Action details: {interrupt_info.value}")
        
        # Simulate human interaction block
        print("\n👥 [Human Feedback Input]: Approving the request...")
        
        # Format the resume payload required by HumanInTheLoopMiddleware
        resume_command = Command(resume={"decisions": [{"type": "approve"}]})
        
        print("\n🔄 Resuming execution with human decision...")
        # Invoke again passing the resume Command inside the same thread config
        response = agent_hitl.invoke(resume_command, config=config)

# --- 5. Print final response ---
print("\n✨ Final Agent Answer:")
print(response["messages"][-1].content)

🚀 Starting Agent Execution...

✨ Final Agent Answer:

